# Image filters on the final dataset

Runs the NSFW and watermark detectors chosen after the two benchmark notebooks (`filters_nsfw_benchmark`,
`filters_watermark_benchmark`) on all 56,568 images of the final dataset (`editjudge-data`: images/, pairs.jsonl),
and writes one decision per image. With several models of a task, their votes are combined by `METHOD`.

Speed and memory: every model is split into two shards, one per T4, so both GPUs work on it at once; images are
decoded on the CPU while the GPU runs; every model runs in its own process and environment. Rough time on 2 x T4
(from the benchmark throughput): MobileNetV2 or ConvNeXt about 5-10 min, Falconsai about 8 min, NeMo about 35 min.

Then, on the coordinator's computer: put `decisions.jsonl` of each task into `work/filters/<task>/`, run
`python -m src.rank export` (a flagged edit loses its pairs; a sample is dropped when its original is flagged or
fewer than 2 edits are left) and `python -m src.process pack`, and upload the new Kaggle dataset.

Before running: input `editjudge-data`; GPU T4 x2, internet on; push the code; get the coordinator's approval.


In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"
os.environ["EJ_WORK_DIR"] = "/tmp/work"    # benchmark images and labels: never in the saved output
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs" # one Python environment per group of conflicting packages
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # public models need no token: hide the rate-limit notice
    os.environ["HF_HUB_VERBOSITY"] = "error"

PREVIOUS = None                            # e.g. "/kaggle/input/<this notebook's previous version>/out"
if PREVIOUS:
    shutil.copytree(f"{PREVIOUS}/filters", f"{OUT}/filters", dirs_exist_ok=True)

In [ ]:
!pip -q install uv                        # fast creation of the per-group environments (src/filters/envs.py)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m src.filters list

import json
from IPython.display import Markdown, display

def run_filters(*args, quiet=False):
    """python -m src.filters <args>: prints every line as it comes (unless quiet), and returns the output."""
    proc = subprocess.Popen([sys.executable, "-u", "-m", "src.filters", *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        if not quiet:
            print(line, end="", flush=True)
        out.append(line)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, args)
    return "".join(out)

def last_json(text):
    """The JSON object a command prints last (multirun prints progress lines before it)."""
    start = text.rfind("\n{")
    return json.loads(text[start + 1:] if start >= 0 else text)

def show_failures(result):
    for name, why in last_json(result)["failed"].items():
        print("FAILED", name, why)
        log = f"{OUT}/filters/logs/{name}.log"
        if os.path.exists(log):
            print(open(log).read()[-2000:])

In [ ]:
# The choice from the benchmark notebooks: models per task, and how to combine them when there are several
CHOICE = {
    "nsfw": {"models": ["mobilenet"], "method": None},          # e.g. {"models": ["nemo", "mobilenet"], "method": "max"}
    "watermark": {"models": ["convnext"], "method": None},
}
SMOKE = False           # True: 20 images per model shard, to check the run


In [ ]:
import zipfile

summary = {}
for task, choice in CHOICE.items():
    result = run_filters("multirun", "--task", task, "--on", "dataset", "--models", ",".join(choice["models"]),
                *(["--limit", 20] if SMOKE else []))
    show_failures(result)
    summary[task] = last_json(run_filters("vote", "--task", task, "--models", ",".join(choice["models"]),
                                  *(["--method", choice["method"]] if choice["method"] else [])))
print(json.dumps(summary, indent=1))

with zipfile.ZipFile(f"{OUT}/filter_decisions.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for task in CHOICE:
        archive.write(f"{OUT}/filters/{task}/decisions.jsonl", f"{task}/decisions.jsonl")
print("download out/filter_decisions.zip")